# Going further: laser linewidth, noise and chirp {#sec-linewidth}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/linewidth-noise.ipynb){.colab-link} &nbsp;Run this chapter's code in Google Colab (no installation).

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippevelha/Optoelectronics-teaching/blob/main/chapter4/notebooks/linewidth-noise.ipynb)

> Generated from the Quarto companion. The browser sliders of the website are not included; see `interactive_explorers.ipynb` for ipywidgets versions.

In [ ]:
# --- setup: works locally and in Google Colab ---------------------------
import sys
if 'google.colab' in sys.modules:
    import urllib.request
    urllib.request.urlretrieve('https://raw.githubusercontent.com/philippevelha/Optoelectronics-teaching/main/chapter4/laserlib.py', 'laserlib.py')
else:
    sys.path.insert(0, '..')                 # laserlib.py is in the parent folder
def ojs_define(**kw): pass                   # only used by the Quarto website
# ----------------------------------------------------------------------
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import laserlib as lz
lz.use_style()

*Not in the slides. The tables of slides 143 and 149 quote linewidths from 50 kHz to 0.1 nm, and the notes of slide 163 mention
"chirping", but the chapter does not explain what limits the linewidth of a single-mode laser, or why directly modulated lasers
change colour. This chapter connects the rate equations (@sec-ld-equation, @sec-modulation) with coherence (Ch. 1) and fiber
dispersion (Ch. 2).*

## Key relations

**Linewidth.** Even a perfectly single-mode laser is not monochromatic. Each spontaneous photon that enters the lasing mode adds
a field of random phase to the coherent field. The phase performs a random walk, and the line acquires a Lorentzian shape whose
width is the **Schawlow–Townes** linewidth, modified for a laser above threshold:

$$
\Delta\nu=\frac{\pi h\nu\,(\Delta\nu_c)^2\,n_{sp}}{P_\text{out}}\,(1+\alpha_H^2),\qquad \Delta\nu_c=\frac{1}{2\pi\tau_{ph}}.
$$

The linewidth falls as $1/P$ and as $\tau_{ph}^2$: a long, low-loss cavity helps quadratically. In semiconductor lasers the
**linewidth enhancement factor** $\alpha_H=-\frac{4\pi}{\lambda}\frac{dn/dN}{dg/dN}\approx3$–6 (Henry, 1982) multiplies the width
by $1+\alpha_H^2\approx10$–40: every fluctuation of the carrier density changes the refractive index and therefore the phase.

**Intensity noise.** The same spontaneous events make the power fluctuate. The relative intensity noise (RIN) peaks at the
relaxation-oscillation frequency $f_r$ (@sec-modulation) and decreases at high power; its floor is the shot-noise limit
$\text{RIN}_\text{SQL}=2h\nu/P$.

**Chirp.** When the current changes, the carrier density changes, and with it the index. The optical frequency follows the
power transients (Koch and Linke):

$$
\Delta\nu(t)=\frac{\alpha_H}{4\pi}\left[\frac{d\ln P}{dt}+\kappa P\right],
$$

where the first term is **transient** chirp and the second, from gain compression, **adiabatic** chirp.

## Linewidth calculator

```{ojs}
//| echo: false
lwPresets = ({
  "DFB laser diode, 1550 nm":           {lam: 1550, L: 400, ng: 3.6, at: 30, P: 10, nsp: 1.5, aH: 4},
  "FP laser diode (one mode), 1310 nm": {lam: 1310, L: 300, ng: 3.6, at: 60, P: 5, nsp: 1.5, aH: 4},
  "external-cavity diode laser":        {lam: 1550, L: 30000, ng: 1.2, at: 1.0, P: 10, nsp: 1.5, aH: 4},
  "VCSEL, 850 nm":                      {lam: 850, L: 2, ng: 3.6, at: 60, P: 0.5, nsp: 1.5, aH: 3},
  "He–Ne, 632.8 nm (slide 73)":         {lam: 632.8, L: 400000, ng: 1.0, at: 0.00114, P: 2.5, nsp: 1.0, aH: 0}
})
viewof lwp = Inputs.select(Object.keys(lwPresets), {label: "start from"})
```

```{ojs}
//| echo: false
viewof lw = {
  const p = lwPresets[lwp];
  return Inputs.form({
    lam: Inputs.number({value: p.lam, label: "wavelength (nm)"}),
    L: Inputs.number({value: p.L, label: "cavity length (µm)"}),
    ng: Inputs.number({value: p.ng, label: "group index"}),
    at: Inputs.number({value: p.at, label: "total loss α_t (cm⁻¹)"}),
    P: Inputs.range([0.01, 100], {value: p.P, step: 0.01, label: "output power (mW)"}),
    nsp: Inputs.range([1, 3], {value: p.nsp, step: 0.05, label: "n_sp"}),
    aH: Inputs.range([0, 8], {value: p.aH, step: 0.1, label: "linewidth enhancement α_H"})
  });
}
```

```{ojs}
//| echo: false
lwOut = {
  const c = 299792458, h = 6.62607015e-34, nu = c/(lw.lam*1e-9), tph = lw.ng/(c*lw.at*100), dnc = 1/(2*Math.PI*tph);
  const dnu = Math.PI*h*nu*dnc**2*lw.nsp*(1 + lw.aH**2)/(lw.P*1e-3);
  return {tph, dnc, dnu, lc: c/(Math.PI*dnu), dlam: (lw.lam*1e-9)**2*dnu/c, rin: 10*Math.log10(2*h*nu/(lw.P*1e-3))};
}
html`<div class="result-box">τ<sub>ph</sub> = ${lwOut.tph < 1e-9 ? (lwOut.tph*1e12).toFixed(2) + " ps" : (lwOut.tph*1e9).toFixed(2) + " ns"} · cold-cavity width Δν<sub>c</sub> = ${(lwOut.dnc/1e9).toPrecision(3)} GHz<br>
linewidth Δν = <b>${lwOut.dnu > 1e6 ? (lwOut.dnu/1e6).toPrecision(3) + " MHz" : lwOut.dnu > 1e3 ? (lwOut.dnu/1e3).toPrecision(3) + " kHz" : lwOut.dnu.toPrecision(3) + " Hz"}</b>
(Δλ = ${(lwOut.dlam*1e15).toPrecision(3)} fm) · coherence length c/πΔν = <b>${lwOut.lc > 1e3 ? (lwOut.lc/1e3).toPrecision(3) + " km" : lwOut.lc.toPrecision(3) + " m"}</b> ·
shot-noise RIN floor 2hν/P = ${lwOut.rin.toFixed(1)} dB/Hz</div>`
```

The presets reproduce the orders of magnitude of slides 143 and 149: about 1–10 MHz for DFB lasers, tens of kHz for external
cavities, tens of MHz for VCSELs. The He–Ne value (millihertz) is a theoretical limit; real gas lasers are limited to
kHz–MHz by mirror vibrations and thermal drift, a technical noise that the formula ignores.

In [ ]:
#| label: fig-lw
#| fig-cap: "Schawlow–Townes–Henry linewidth versus output power for a 400 µm DFB laser (α_t = 30 cm⁻¹, n_g = 3.6, n_sp = 1.5) and three values of α_H. Real lasers follow 1/P at low power and saturate ('linewidth floor') at a few hundred kHz at high power."
P = np.logspace(-4, -1, 100)
tph = 3.6/(lz.c0*3000); dnc = 1/(2*np.pi*tph)
fig, ax = plt.subplots(figsize=(7.2, 3.6))
for aH in (0, 3, 6):
    ax.loglog(P*1e3, lz.schawlow_townes(P, lz.c0/1.55e-6, dnc, 1.5, aH), label=f"α_H = {aH}")
ax.set(xlabel="output power (mW)", ylabel="linewidth Δν (Hz)"); ax.legend(fontsize=8); plt.show()

## Chirp of a directly modulated laser

In [ ]:
#| label: fig-chirp
#| fig-cap: "Power and transient chirp (α_H = 5) of the slide-131 laser switched from 1.1 I_th to 3 I_th. The frequency swings by about ±15 GHz during the relaxation oscillations. In 20 km of standard fiber (D = 17 ps nm⁻¹ km⁻¹), a 10 GHz excursion at 1550 nm (0.08 nm) delays the leading edge relative to the rest by about 27 ps, a quarter of a 10 Gb/s bit."
V = 250e-6*5e-6*0.15e-6
t, n, N = lz.ld_step_response(1.1*24e-3, 72e-3, 24e-3, 2.5e-9, 2.16e-12, V, t_end=2e-9, n_t=8000)
k = lz.ld_power(1 + 24e-3, 24e-3, 2.16e-12, 0.32, 3.6, 860e-9, 250e-6)
P = k*lz.qe*V*N/2.16e-12
chirp = 5/(4*np.pi)*np.gradient(np.log(P), t)
fig, ax = plt.subplots(2, 1, figsize=(7.5, 4.6), sharex=True)
ax[0].plot(t*1e9, P*1e3, color=lz.PALETTE[7]); ax[0].set(ylabel="P_o (mW)")
ax[1].plot(t*1e9, chirp/1e9, color=lz.PALETTE[0]); ax[1].set(xlabel="time (ns)", ylabel="Δν (GHz)")
plt.tight_layout(); plt.show()
print(f"peak frequency excursion: {chirp.max()/1e9:.1f} GHz, {-chirp.min()/1e9:.1f} GHz")

Chirp converts the clean intensity pulse into a pulse whose leading and trailing edges have different wavelengths. Fiber
dispersion (Ch. 2) then moves them apart. This is why 1550 nm links above 2.5 Gb/s over more than a few tens of km use a CW DFB
laser followed by an external modulator (Mach–Zehnder or electro-absorption, Ch. 6), whose chirp is close to zero.

::: {.callout-tip collapse="true"}
## Going further
- **Measure a linewidth.** The delayed self-heterodyne method beats the laser with a copy of itself delayed by more than its
  coherence time and frequency-shifted by an acousto-optic modulator. Simulate it: generate a phase random walk with diffusion
  coefficient $2\pi\Delta\nu$, compute the beat spectrum with `numpy.fft`, and check that it is a Lorentzian of width $2\Delta\nu$.
- **Narrow-linewidth lasers.** Self-injection locking to a high-Q microresonator brings a DFB laser from MHz to Hz-level
  linewidths. Use the $\tau_{ph}^2$ scaling to estimate the Q factor needed.
- **Phase noise and coherent communications.** QPSK and 16-QAM coherent receivers (Ch. 2) tolerate a linewidth × symbol-time
  product of about $10^{-4}$. Which lasers of the calculator are suitable at 32 GBd?
:::

::: {.callout-note collapse="true"}
## Complements
- C. H. Henry, "Theory of the linewidth of semiconductor lasers", *IEEE J. Quantum Electron.* 18, 259 (1982).
- T. L. Koch and R. A. Linke, "Effect of nonlinear gain reduction on semiconductor laser wavelength chirping", *Appl. Phys. Lett.*
  48, 613 (1986).
- Coldren *et al.*, ch. 5 (RIN, linewidth, chirp); Agrawal, *Fiber-Optic Communication Systems*, sec. 3.5 and 5.4.
:::

## Try it yourself

1. A DFB laser has a 2 MHz linewidth at 5 mW. What linewidth do you expect at 20 mW? What is the coherence length?
2. By what factor must the cavity length grow to reduce the linewidth 100 times, at constant loss per unit length? At constant
   mirror loss?
3. Using the chirp figure, estimate the extra dispersion-induced broadening after 80 km. Is a 10 Gb/s link possible?
4. Why does the He–Ne laser have $\alpha_H\approx0$?